# Программирование для всех<br>(основы работы с Python)

*Тамбовцева А.А.*

## Практикум 6.1. Работа с API на примере ВКонтакте: извлечение постов со страницы сообщества

Загрузим необходимые для работы библиотеки, модули и функции:

* модуль `requests` для отправки запроса к серверу и получения ответа;
* библиотека `pandas` для работы с таблицами.

In [1]:
import requests
import pandas as pd

**ВАЖНО.** При отправке запросов через `resuests` из-за разных настроек в разных версиях модуля могут возникать ошибки, в частности, ошибка о непроверенных сертификатах. Отправка запросов через `requests` подразумевает проверку SSL-сертификата, то есть проверку того, является ли сервер настоящим (не двойник от мошенников), плюс, есть ли у сервера действующий сертификат, гарантирующий безопасную передачу данных.

Поскольку мы не будем заходить на непроверенные сайты и передавать ключ доступа в виде заголовков (есть такой параметр `headers` в запросах), далее мы выключим автоматическую проверку безопасности сертификатов, а чтобы при этом каждый раз нам не выводилось сообщение с предупреждением `InsecureRequestWarning`, отключим заодно и уведомления такого вида:

In [2]:
from urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)

Ранее мы получили доступ к API ВКонтакте через **сервисный ключ** приложения (если нет – см инструкцию [здесь](https://github.com/allatambov/PyAll26/blob/main/vk-api-access.md)). Этот сервисный ключ нужно добавить ниже – сохранить в переменную `service_key`:

In [3]:
# скопируйте внутрь кавычек ключ, набор
# букв и цифра

service_key = ""

В [документации](https://dev.vk.com/ru/reference) API зайдем в раздел «Формат запросов» и изучим синтаксис запроса, чтобы понять код ниже и структуру ссылки в `res.url`.

В разделе [Формат запросов](https://dev.vk.com/ru/api/api-requests) приведен шаблон запроса `GET` для получения данных:

    /method/<имя-API-метода>?<параметр1=значение1>&<параметр2=значение2> 

Что нам дает этот шаблон? Понимание трех базовых вещей:

1. Запрос к API – это ссылка определенного вида, обычная строка с текстом заданной структуры.
2. В такую строку после `method/` нужно подставить название метода, который мы будем использовать (для выгрузки постов со стены – один, для получения информации о пользователях – другой, и так далее), а после названия и символа `?` указать аргументы метода, то есть параметры запроса.
3. Параметры запроса – аргументы метода – подставляются в строку в виде пар `название=значение`, а затем эти пары объединяются через `&`.

Согласно документации (см раздел [Методы](https://dev.vk.com/ru/method)) для получения постов со стены пользователя/сообщества нам нужен метод `wall.get`. У него есть следующие основные параметры:

* `domain` – строка с кратким адресом пользователя/сообщества
* `count` – число постов (максимум 100 за один запрос)
* `filter` – указатель, посты какого типа собирать.

Плюс, есть два параметра, общие для всех методов, без которых результата мы не получим:

* `access_key`: ключ доступа (например, сервисный ключ)
* `v`: версия API.

Почему мы не получим результат без ключа доступа – понятно, а зачем указывать версию API? Очень просто – в разных версиях API набор параметров методов может выглядеть по-разному (добавляются новые, убираются старые, меняются названия или значения по умолчанию).

Ссылка в шаблоне запроса выше – относительная, она начинается с `/method`, абсолютные («полные») ссылки начинаются с `https://`. Зафиксируем ссылку полностью, а параметры запроса, раз они должны быть представлены в виде пар, сохраним в виде словаря.

In [4]:
main_url = "https://api.vk.ru/method/wall.get"

wall_params = {
    "domain": "hseteachers", 
    "count": 100, 
    "filter" : "owner", 
    "access_token": service_key, 
    "v": "5.199"
}

Отправляем запрос вида `GET`, функция `get()` сама склеит ссылку `main_url` и пары из словаря `wall_params`:

In [5]:
# зависит от версии requests, может работать без verify = False

res = requests.get(main_url, params = wall_params, verify = False)

Объект, который мы сохранили в `res`, имеет особый тип `requests.models.Response`, он же ответ на запрос:

In [6]:
print(type(res))
print(res)

<class 'requests.models.Response'>
<Response [200]>


Сам по себе объект полностью мы увидеть не можем, но из него можно извлечь разные **атрибуты** – характеристики объекта. Например, статус ответа:

In [7]:
print(res.status_code)

200


> Код 200 означает, что запрос получен, обработан, и ответ на него получен. Другими словами, у нас нет проблем с доступом к странице (потенциальная проблема со стороны сервера, например, ведущиеся технические работы или отказ в правах доступа) и нет проблем с самим запросом (потенциальная проблема со стороны пользователя, например, «битая» ссылка). Про коды статусов можно почитать в [статье](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).

Или ссылку, которая была сформирована функцией `get()` для отправки запроса:

In [8]:
print(res.url)

https://api.vk.ru/method/wall.get?domain=hseteachers&count=100&filter=owner&access_token=e0c23f42e0c23f42e0c23f4241e386b8040e0c2e0c23f428a1361c9864eeba9aff91757&v=5.199


> В запрос функция подставила ссылку (до `?`) и пары `<параметр1=значение1>`, `<параметр2=значение2>`, объединив их через `&`. Если бы не аргумент `params` в `.get()`, пришлось бы писать свою f-строку и подставлять туда все названия (способ тоже рабочий).

Если перейти по ссылке, откроется страница в браузере с огромным полотном текста – это ответ на запрос в виде большой JSON-строки. Если, используя Python, мы заберем текст «как есть», придется долго ломать голову, как из такого массива символов выбрать нужное:

In [ ]:
# плохая идея
# print(res.text)

А вот преобразовать эту JSON-строку через метод `.json()` в структуру данных Python – идея хорошая:

In [9]:
res_dict = res.json()

## Сюжет 1. Забираем 100 постов со стены сообщества и очищаем данные

Выше мы преобразовали JSON-строку с ответом на запрос в один большой словарь (десериализовали JSON в Python). Теперь мы можем поработать с этим словарем и извлечь из него необходимые данные!

### Задача 1

Получите, используя словарь `res_dict`, список словарей `items`, где один словарь соответствует одному посту.  

In [10]:
items = res_dict["response"]["items"]
print(len(items)) # должно быть 100 постов

100


### Задача 2

Напишите функцию `clean_posts()`, которая принимает на вход один словарь с характеристиками поста и возвращает список из следующих значений:

* id поста
* дата публикации 
* текст поста
* число лайков
* число просмотров

Примените функцию к каким-нибудь элементам в `items` и проверьте, что она работает. Как вы думаете, может ли у поста какая-то характеристика отсутствовать? Как скорректировать функцию, чтобы учесть такой сценарий?

In [12]:
# выберем один пост – один словарь для примера,
# чтобы понять, что и как извлекать,
# отсутствовать может views,
# счетчик просмотров появился в 2017 году

d = items[-1]
print(d)

{'inner_type': 'wall_wallpost', 'donut': {'is_donut': False}, 'comments': {'count': 0}, 'marked_as_ads': 0, 'hash': 'IOvjZf7I6YFyGLyPQw', 'type': 'post', 'push_subscription': {'is_subscribed': False}, 'attachments': [], 'date': 1738680378, 'from_id': -63442801, 'id': 40433, 'likes': {'can_like': 0, 'count': 25, 'user_likes': 0}, 'reaction_set_id': 'reactions', 'reactions': {'count': 25, 'items': [{'id': 0, 'count': 24}, {'id': 1, 'count': 1}]}, 'owner_id': -63442801, 'post_type': 'post', 'reposts': {'count': 3}, 'text': '"Я там не жил, не знаю".\n\n#Юрфак #Преподаватель_ВШЭ\n#игпзс', 'views': {'count': 5735}}


In [13]:
def clean_posts(d):
    id_ = d["id"]
    date = d["date"]
    text = d["text"]
    likes = d["likes"]["count"]

    if "views" in d:
        views = d["views"]["count"]
    else:
        views = None
    return [id_, date, text, likes, views]

In [14]:
# применяем к отдельным элементам

print(clean_posts(items[-1]))
print(clean_posts(items[1]))

[40433, 1738680378, '"Я там не жил, не знаю".\n\n#Юрфак #Преподаватель_ВШЭ\n#игпзс', 25, 5735]
[40741, 1791795840, '"Sometimes solving these problems requires your tears, sweat, and blood, among other things"\n\n(с) Oleg Borisovich\n\n[club63442801|@hseteachers] | [https://vk.ru/wall-63442801_40706|Прислать пост]', 5, 685]


### Задача 3

Напишите код (используя функцию из задачи 2), который сохранит в список `final` характеристики только тех постов, которые не содержат прикрепленных изображений и иных файлов.

In [15]:
# если нет прикрепленных файлов - в записи 
# с ключом attachements пустой список длины 0

final = []
for d in items:
    if len(d["attachments"]) == 0:
        final.append(clean_posts(d))
print(len(final))

99


Итак, если `final` содержит списки с характеристиками постов (один список = один пост), его можно преобразовать в таблицу, объект типа `pandas DataFrame` (один список = одна строка в таблице):

In [16]:
df = pd.DataFrame(final, columns = ["id", "date", "text", "likes", "views"])
df

,id,date,text,likes,views
0,40741,1791795840,"""Sometimes solving these problems requires you...",5,685
1,40739,1791734653,"""Ну если вы из этих конечно"" \n(имелись ввиду ...",1,862
2,40736,1791716622,«Вам нужен линукс?»\n\n@ Булгаков Станислав Ал...,0,703
3,40734,1791538673,"""За базар нужно отвечать!""\n\n• Андрей Пунин\n...",2,1183
4,40731,1789657800,В техе это означается как \cap … шапка\n\n(c) ...,3,1382
...,...,...,...,...,...
94,40454,1741778040,"И мы не жалкие филологи, а мы гордые любители ...",23,4045
95,40450,1741771870,«Вот вы смотрели фильмы с Брюсом Уиллисом? Вот...,48,4481
96,40440,1741195327,"С: *Сдаёт лабу по физике*\nП: Вот скажи, прогр...",27,4307
97,40437,1740060710,"*На физ-ре, преподаватель л/а Дальнопилус Арсе...",33,4729


> Формат даты, который используется здесь в API, это UNIX-формат, он же POSIX-формат. Отметки времени в таком формате – это число секунд с 1 января 1970 года.

Преобразуем дату в более привычный формат:

In [17]:
# как раз переводим из числа секунд c 01.01.1970
# unit = "s", от seconds

df["date"] = pd.to_datetime(df["date"], unit="s")
df

,id,date,text,likes,views
0,40741,2026-10-12 09:04:00,"""Sometimes solving these problems requires you...",5,685
1,40739,2026-10-11 16:04:13,"""Ну если вы из этих конечно"" \n(имелись ввиду ...",1,862
2,40736,2026-10-11 11:03:42,«Вам нужен линукс?»\n\n@ Булгаков Станислав Ал...,0,703
3,40734,2026-10-09 09:37:53,"""За базар нужно отвечать!""\n\n• Андрей Пунин\n...",2,1183
4,40731,2026-09-17 15:10:00,В техе это означается как \cap … шапка\n\n(c) ...,3,1382
...,...,...,...,...,...
94,40454,2025-03-12 11:14:00,"И мы не жалкие филологи, а мы гордые любители ...",23,4045
95,40450,2025-03-12 09:31:10,«Вот вы смотрели фильмы с Брюсом Уиллисом? Вот...,48,4481
96,40440,2025-03-05 17:22:07,"С: *Сдаёт лабу по физике*\nП: Вот скажи, прогр...",27,4307
97,40437,2025-02-20 14:11:50,"*На физ-ре, преподаватель л/а Дальнопилус Арсе...",33,4729


Можем выгрузить в CSV-файл или файл Excel:

In [18]:
df.to_csv("hseposts_100.csv")
df.to_excel("hseposts_100.xlsx")

### Сюжет 2. Забираем все посты со стены сообщества, создаем и обрабатываем таблицу

Ограничения данного API таковы, что за один раз мы можем выгрузить только 100 постов. Хорошие новости: каждый раз при выгрузке мы можем начинать с того поста, на котором закончили, то есть сначала выгрузить первые 100 постов, потом – следующие 100 постов, и так до тех пор, пока не заполучим все. Обеспечивается это аргументом `offset` в методе `wall.get` в API.

Тем не менее, остаются ограничения на число запросов в секунду. Для надежности (во избежание блокировки) импортируем функцию `sleep` из модуля `time`, которая позволит выставлять задержку в одну секунду после выгрузки очередных 100 постов.

In [19]:
from time import sleep

### Задача 1

Сохраните общее число постов в переменную `nposts`. Посчитайте, сколько раз нужно будет выполнить выгрузку по 100 постов, чтобы собрать все тексты, и сохраните значение в переменную `iterate`.

In [22]:
nposts = res_dict["response"]["count"]
print(nposts)

10201


In [23]:
# поделим nposts нацело на 100 и добавим 1

iterate = nposts // 100 + 1
print(iterate)

103


### Задача 2

Используя информацию об аргументе `offset` и блоки кода ниже, выгрузите и сохраните в список `items_all` данные по всем постам на стене сообщества.

**Подсказка:** чтобы расширять список правильным образом, используйте метод `.extend()`, а не `.append()`, он добавляет не один элемент, а сразу несколько.

In [24]:
wall_params_long = {
    "domain": "hseteachers", 
    "count": 100, 
    "offset" : 0,
    "filter" : "owner", 
    "access_token": service_key, 
    "v": "5.199"
}

items_all = []

for i in range(iterate):
    res_long = requests.get(main_url, params = wall_params_long)
    res_dict_long = res_long.json()
    data_long = res_dict_long['response']['items']
    
    ### YOUR CODE HERE ###
    items_all.extend(data_long)
    wall_params_long["offset"] += 100
    
    sleep(1)
    print(i, end = " ... ")

0 ... 1 ... 2 ... 3 ... 4 ... 5 ... 6 ... 7 ... 8 ... 9 ... 10 ... 11 ... 12 ... 13 ... 14 ... 15 ... 16 ... 17 ... 18 ... 19 ... 20 ... 21 ... 22 ... 23 ... 24 ... 25 ... 26 ... 27 ... 28 ... 29 ... 30 ... 31 ... 32 ... 33 ... 34 ... 35 ... 36 ... 37 ... 38 ... 39 ... 40 ... 41 ... 42 ... 43 ... 44 ... 45 ... 46 ... 47 ... 48 ... 49 ... 50 ... 51 ... 52 ... 53 ... 54 ... 55 ... 56 ... 57 ... 58 ... 59 ... 60 ... 61 ... 62 ... 63 ... 64 ... 65 ... 66 ... 67 ... 68 ... 69 ... 70 ... 71 ... 72 ... 73 ... 74 ... 75 ... 76 ... 77 ... 78 ... 79 ... 80 ... 81 ... 82 ... 83 ... 84 ... 85 ... 86 ... 87 ... 88 ... 89 ... 90 ... 91 ... 92 ... 93 ... 94 ... 95 ... 96 ... 97 ... 98 ... 99 ... 100 ... 101 ... 102 ... 

In [25]:
print(len(items_all))

10201


Ниже написан код, который переводит список `items_all` в датафрейм `df_full`. Так как здесь данные мы предварительно не очищали и не фильтровали, столбцов много, и некоторые из них со словарями внутри. Выберем из `full` только нужные столбцы и сохраним в `df_small`.

In [26]:
df_full = pd.DataFrame(items_all)
df_small = df_full[["id", "date", "text", "likes", "views"]]
df_small

,id,date,text,likes,views
0,40706,1788163475,Прислать пост нам совершенно анонимно можно че...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 2235}
1,40741,1791795840,"""Sometimes solving these problems requires you...","{'can_like': 0, 'count': 5, 'user_likes': 0}",{'count': 686}
2,40739,1791734653,"""Ну если вы из этих конечно"" \n(имелись ввиду ...","{'can_like': 0, 'count': 1, 'user_likes': 0}",{'count': 863}
3,40736,1791716622,«Вам нужен линукс?»\n\n@ Булгаков Станислав Ал...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 704}
4,40734,1791538673,"""За базар нужно отвечать!""\n\n• Андрей Пунин\n...","{'can_like': 0, 'count': 2, 'user_likes': 0}",{'count': 1184}
...,...,...,...,...,...
10196,6,1387918264,по маленькой кругленькой громов\n#Громов_hse\n...,"{'can_like': 0, 'count': 6, 'user_likes': 0}",NaN
10197,5,1387917399,Все глоки суть куздры (Данько)\n#Данько_hse,"{'can_like': 0, 'count': 19, 'user_likes': 0}",NaN
10198,4,1387917345,с какого бадуна ты это написал? (Самовол)\n#Са...,"{'can_like': 0, 'count': 40, 'user_likes': 0}",NaN
10199,3,1387916061,Синдром яндекса (Шаповалов И. А)\n#Шаповалов_hse,"{'can_like': 0, 'count': 24, 'user_likes': 0}",NaN


Осталось разобраться со словарями внутри ячеек таблицы!

### Задача 3

В `pandas` есть метод `.apply()`, который позволяет применить какую-то функцию ко всем ячейкам в столбце.

In [27]:
# применяем ко всем ячейкам в столбце функцию len(),
# считаем количество символов

df_small["text"].apply(len)

0        202
1        194
2        166
3        136
4        123
        ... 
10196    108
10197     42
10198     54
10199     47
10200     44
Name: text, Length: 10201, dtype: int64

Его удобно сочетать с короткими анонимными lambda-функциями:

In [28]:
# применяем ко всем ячейкам в столбце lambda-функцию,
# разбиваем на «слова» и считаем количество слов

df_small["text"].apply(lambda x: len(x.split()))

0        30
1        20
2        17
3        11
4        11
         ..
10196    11
10197     6
10198     8
10199     6
10200     5
Name: text, Length: 10201, dtype: int64

Используя метод `.apply()` и lambda-функции, извлеките из `likes` и `views` число лайков и просмотров и сохраните их в столбцы `nlikes` и `nviews`. 

In [29]:
# с лайками просто – все ячейки заполнены, 
# везде внутри словари, забираем значение по ключу count

df_small["nlikes"] = df_small["likes"].apply(lambda x: x["count"])
df_small

,id,date,text,likes,views,nlikes
0,40706,1788163475,Прислать пост нам совершенно анонимно можно че...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 2235},0
1,40741,1791795840,"""Sometimes solving these problems requires you...","{'can_like': 0, 'count': 5, 'user_likes': 0}",{'count': 686},5
2,40739,1791734653,"""Ну если вы из этих конечно"" \n(имелись ввиду ...","{'can_like': 0, 'count': 1, 'user_likes': 0}",{'count': 863},1
3,40736,1791716622,«Вам нужен линукс?»\n\n@ Булгаков Станислав Ал...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 704},0
4,40734,1791538673,"""За базар нужно отвечать!""\n\n• Андрей Пунин\n...","{'can_like': 0, 'count': 2, 'user_likes': 0}",{'count': 1184},2
...,...,...,...,...,...,...
10196,6,1387918264,по маленькой кругленькой громов\n#Громов_hse\n...,"{'can_like': 0, 'count': 6, 'user_likes': 0}",NaN,6
10197,5,1387917399,Все глоки суть куздры (Данько)\n#Данько_hse,"{'can_like': 0, 'count': 19, 'user_likes': 0}",NaN,19
10198,4,1387917345,с какого бадуна ты это написал? (Самовол)\n#Са...,"{'can_like': 0, 'count': 40, 'user_likes': 0}",NaN,40
10199,3,1387916061,Синдром яндекса (Шаповалов И. А)\n#Шаповалов_hse,"{'can_like': 0, 'count': 24, 'user_likes': 0}",NaN,24


In [30]:
# с просмотрами сложнее – напишем отдельную функцию,
# которая предусматривает наличие пропуска NaN вместо словаря,
# если тип значения – dict, забираем count,
# если нет, оставляем как есть пустое значение

def get_count(x):
    if type(x) == dict:
        y = x["count"]
    else:
        y = x
    return y

In [32]:
df_small["nviews"] = df_small["views"].apply(get_count)
df_small

,id,date,text,likes,views,nlikes,nviews
0,40706,1788163475,Прислать пост нам совершенно анонимно можно че...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 2235},0,2235.0
1,40741,1791795840,"""Sometimes solving these problems requires you...","{'can_like': 0, 'count': 5, 'user_likes': 0}",{'count': 686},5,686.0
2,40739,1791734653,"""Ну если вы из этих конечно"" \n(имелись ввиду ...","{'can_like': 0, 'count': 1, 'user_likes': 0}",{'count': 863},1,863.0
3,40736,1791716622,«Вам нужен линукс?»\n\n@ Булгаков Станислав Ал...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 704},0,704.0
4,40734,1791538673,"""За базар нужно отвечать!""\n\n• Андрей Пунин\n...","{'can_like': 0, 'count': 2, 'user_likes': 0}",{'count': 1184},2,1184.0
...,...,...,...,...,...,...,...
10196,6,1387918264,по маленькой кругленькой громов\n#Громов_hse\n...,"{'can_like': 0, 'count': 6, 'user_likes': 0}",NaN,6,NaN
10197,5,1387917399,Все глоки суть куздры (Данько)\n#Данько_hse,"{'can_like': 0, 'count': 19, 'user_likes': 0}",NaN,19,NaN
10198,4,1387917345,с какого бадуна ты это написал? (Самовол)\n#Са...,"{'can_like': 0, 'count': 40, 'user_likes': 0}",NaN,40,NaN
10199,3,1387916061,Синдром яндекса (Шаповалов И. А)\n#Шаповалов_hse,"{'can_like': 0, 'count': 24, 'user_likes': 0}",NaN,24,NaN


> Раз в столбце `nviews` есть пропуски, он всегда автоматически будет типа `float`, даже если все остальные числа – целые, так что дробный формат у числа просмотров выше – это нормально.

**Дополнительно.** Можно было это через lambda-функцию сделать в одну строчку, но там `if-else` раскрывается с конца (по аналогии со списковыми включениями с `for`):

In [33]:
df_small["nviews"] = df_small["views"].apply(lambda x: x["count"] if type(x) == dict else x)
df_small

,id,date,text,likes,views,nlikes,nviews
0,40706,1788163475,Прислать пост нам совершенно анонимно можно че...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 2235},0,2235.0
1,40741,1791795840,"""Sometimes solving these problems requires you...","{'can_like': 0, 'count': 5, 'user_likes': 0}",{'count': 686},5,686.0
2,40739,1791734653,"""Ну если вы из этих конечно"" \n(имелись ввиду ...","{'can_like': 0, 'count': 1, 'user_likes': 0}",{'count': 863},1,863.0
3,40736,1791716622,«Вам нужен линукс?»\n\n@ Булгаков Станислав Ал...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 704},0,704.0
4,40734,1791538673,"""За базар нужно отвечать!""\n\n• Андрей Пунин\n...","{'can_like': 0, 'count': 2, 'user_likes': 0}",{'count': 1184},2,1184.0
...,...,...,...,...,...,...,...
10196,6,1387918264,по маленькой кругленькой громов\n#Громов_hse\n...,"{'can_like': 0, 'count': 6, 'user_likes': 0}",NaN,6,NaN
10197,5,1387917399,Все глоки суть куздры (Данько)\n#Данько_hse,"{'can_like': 0, 'count': 19, 'user_likes': 0}",NaN,19,NaN
10198,4,1387917345,с какого бадуна ты это написал? (Самовол)\n#Са...,"{'can_like': 0, 'count': 40, 'user_likes': 0}",NaN,40,NaN
10199,3,1387916061,Синдром яндекса (Шаповалов И. А)\n#Шаповалов_hse,"{'can_like': 0, 'count': 24, 'user_likes': 0}",NaN,24,NaN


Можем, как и в первой части, переделать столбец с датой-временем:

In [34]:
df_small["date"] = pd.to_datetime(df_small["date"], unit="s")
df_small

,id,date,text,likes,views,nlikes,nviews
0,40706,2026-08-31 08:04:35,Прислать пост нам совершенно анонимно можно че...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 2235},0,2235.0
1,40741,2026-10-12 09:04:00,"""Sometimes solving these problems requires you...","{'can_like': 0, 'count': 5, 'user_likes': 0}",{'count': 686},5,686.0
2,40739,2026-10-11 16:04:13,"""Ну если вы из этих конечно"" \n(имелись ввиду ...","{'can_like': 0, 'count': 1, 'user_likes': 0}",{'count': 863},1,863.0
3,40736,2026-10-11 11:03:42,«Вам нужен линукс?»\n\n@ Булгаков Станислав Ал...,"{'can_like': 0, 'count': 0, 'user_likes': 0}",{'count': 704},0,704.0
4,40734,2026-10-09 09:37:53,"""За базар нужно отвечать!""\n\n• Андрей Пунин\n...","{'can_like': 0, 'count': 2, 'user_likes': 0}",{'count': 1184},2,1184.0
...,...,...,...,...,...,...,...
10196,6,2013-12-24 20:51:04,по маленькой кругленькой громов\n#Громов_hse\n...,"{'can_like': 0, 'count': 6, 'user_likes': 0}",NaN,6,NaN
10197,5,2013-12-24 20:36:39,Все глоки суть куздры (Данько)\n#Данько_hse,"{'can_like': 0, 'count': 19, 'user_likes': 0}",NaN,19,NaN
10198,4,2013-12-24 20:35:45,с какого бадуна ты это написал? (Самовол)\n#Са...,"{'can_like': 0, 'count': 40, 'user_likes': 0}",NaN,40,NaN
10199,3,2013-12-24 20:14:21,Синдром яндекса (Шаповалов И. А)\n#Шаповалов_hse,"{'can_like': 0, 'count': 24, 'user_likes': 0}",NaN,24,NaN


И выгрузить результат в CSV или Excel:

In [35]:
df_small.to_csv("hseposts_all.csv")
df_small.to_excel("hseposts_all.xlsx")